# DeBERTa Stance Classification Evaluation Pipeline

This notebook evaluates a **fine-tuned DeBERTa-v3 model** on benchmark argument mining data, focusing on **Stance Classification** between claims and premises.

### Evaluation Components:

- **Model**: DeBERTa-v3-base fine-tuned on stance classification.
- **Task**: Classify the stance (pro/con) between a claim and a premise.
- **Data**: Benchmark dataset (claim-premise pairs with stance labels).
- **Metrics**:
  - Accuracy
  - F1 Score
  - Classification Report
  - Error Analysis

---

### Pipeline Steps:

1. **Tokenizer & Model Load**  
   Load the DeBERTa-v3 tokenizer and classification head with 2 labels.

2. **Benchmark Data Preparation**  
   Load and format claim-premise pairs with stance labels.

3. **Prediction Loop**  
   - Encode each input pair.
   - Run through model.
   - Collect predictions and true labels.

4. **Evaluation**  
   - Compute accuracy and F1.
   - Generate confusion matrix and classification report.
   - Save error cases (misclassifications) to `error_analysis.csv`.

---



In [1]:
# Install required packages
! pip install transformers datasets scikit-learn tqdm --quiet

# Clone the Argument Mining repo for benchmark data
!git clone https://github.com/Horizontal-Labs/Argument-Mining.git

# Add the repo to the system path
import sys
sys.path.append("/content/Argument-Mining")


Cloning into 'Argument-Mining'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 154 (delta 68), reused 106 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 227.04 KiB | 3.29 MiB/s, done.
Resolving deltas: 100% (68/68), done.


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import pandas as pd
from typing import List, Dict, Any
import logging
from tqdm import tqdm
from pathlib import Path
from .quality_data import data as benchmark_data
from .db import get_session
from .models import ADU, Relationship

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [3]:
def prepare_benchmark_data() -> List[Dict[str, Any]]:
    with get_session() as session:
        claim_ids = list(benchmark_data.keys())
        premise_ids = list({pid for pids in benchmark_data.values() for pid in pids})

        claims = session.query(ADU).filter(ADU.id.in_(claim_ids)).all()
        premises = session.query(ADU).filter(ADU.id.in_(premise_ids)).all()

        claims_by_id = {c.id: c for c in claims}
        premises_by_id = {p.id: p for p in premises}

        rows = (
            session.query(Relationship.from_adu_id, Relationship.to_adu_id, Relationship.category)
            .filter(
                Relationship.to_adu_id.in_(claim_ids),
                Relationship.from_adu_id.in_(premise_ids)
            ).all()
        )
        category_lookup = {(from_id, to_id): cat for from_id, to_id, cat in rows}

        examples = []
        for claim_id, premise_list in benchmark_data.items():
            claim = claims_by_id.get(claim_id)
            for pid in premise_list:
                premise = premises_by_id.get(pid)
                if claim and premise:
                    stance = category_lookup.get((pid, claim_id), None)
                    if stance in ["stance_pro", "stance_con"]:
                        examples.append({
                            "claim_text": claim.text,
                            "premise_text": premise.text,
                            "stance": 1 if stance == "stance_pro" else 0
                        })
        return examples


In [22]:
def evaluate_stance_classification(model_path: str):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2).to(device)
    model.eval()

    examples = prepare_benchmark_data()
    preds = []
    labels = []

    for example in tqdm(examples):
        combined_text = f"[CLS] {example['claim_text']} [SEP] {example['premise_text']} [SEP]"
        inputs = tokenizer(combined_text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = model(**inputs)
            pred = torch.argmax(outputs.logits, dim=1).item()

        preds.append(pred)
        labels.append(example["stance"])

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    report = classification_report(labels, preds, target_names=["stance_con", "stance_pro"])
    cm = confusion_matrix(labels, preds).tolist()

    print("Accuracy:", acc)
    print("F1 Score:", f1)
    print("Classification Report:")
    print(report)
    # Error analysis
    error_data = []
    for i, (pred, gt) in enumerate(zip(preds, labels)):
      if pred != gt:
        error_data.append({
            "index": i,
            "claim_text": examples[i]["claim_text"][:100] + "...",
            "premise_text": examples[i]["premise_text"][:100] + "...",
            "predicted": pred,
            "ground_truth": gt
        })
    pd.DataFrame(error_data).to_csv("error_analysis.csv", index=False)
    print(f"Error analysis saved to error_analysis.csv")
    print(f"Found {len(error_data)} errors out of {len(preds)} predictions")


    return {
        "accuracy": acc,
        "f1_score": f1,
        "confusion_matrix": cm,
        "report": report
    }


In [14]:
! pip3  install sentencepiece mariadb --quiet


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [6]:

! pip install pymysql


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 2.7 MB/s eta 0:00:00


In [7]:
! apt-get install libmariadb-dev



Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following packages were automatically installed and are no longer required:
  libaom-dev libarmadillo-dev libarpack2-dev libblosc-dev libcfitsio-dev
  libdav1d-dev libde265-dev libfreexl-dev libfyba-dev libgeos-dev
  libgeotiff-dev libgif-dev libhdf4-alt-dev libheif-dev libjson-c-dev
  libkml-dev libkmlconvenience1 libkmlregionator1 libkmlxsd1 liblz4-dev
  libminizip-dev libnetcdf-dev libodbccr2 libogdi-dev libopenjp2-7-dev
  libpoppler-dev libpoppler-private-dev libpq-dev libproj-dev libqhull-dev
  libqhull8.0 libqhullcpp8.0 librttopo-dev libspatialite-dev libsqlite3-dev
  libsuperlu-dev liburiparser-dev libwebp-dev libx265-dev libxerces-c-dev
  unixodbc-dev
Use 'apt autoremove' to remove them.
The following additional packages will be installed:
  libmariadb3 mariadb-common
The following packages will be REMOVED:
  default-libmysqlclient-dev libgdal-dev libmysqlclient-dev
The followin

In [23]:
from transformers import DebertaV2Tokenizer, AutoModelForSequenceClassification
HF_token= ""
def main():
    MODEL_PATH = "microsoft/deberta-v3-base"
    tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_PATH,token=HF_token)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH,token=HF_token)
    print(" Tokenizer and model loaded successfully!")
    evaluate_stance_classification(MODEL_PATH)
if __name__ == "__main__":
    main()



Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


 Tokenizer and model loaded successfully!


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 298/298 [09:24<00:00,  1.89s/it]
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

Accuracy: 0.5
F1 Score: 0.3333333333333333
Classification Report:
              precision    recall  f1-score   support

  stance_con       0.50      1.00      0.67       149
  stance_pro       0.00      0.00      0.00       149

    accuracy                           0.50       298
   macro avg       0.25      0.50      0.33       298
weighted avg       0.25      0.50      0.33       298

Error analysis saved to error_analysis.csv
Found 149 errors out of 298 predictions
